# Exact five-cycle visibility and comparator verification
Run in the repository root. The manuscript proves exactness analytically; these cells independently check the identities numerically.

## Explicit physical non-GMS assemblage

In [ ]:
import numpy as np
from itertools import product
from scipy.optimize import brentq
from pathlib import Path
import csv
c=1/np.sqrt(2); v=1/(3-2*c); lam=2*np.sqrt(2)-2
P={'I':np.eye(2),'X':np.array([[0,1],[1,0]]),'Y':np.array([[0,-1j],[1j,0]]),'Z':np.diag([1,-1])}
def kron(*ops):
 r=np.ones((1,1))
 for op in ops:r=np.kron(r,op)
 return r
bits=list(product([0,1],repeat=5))
G=np.array([(-1)**sum(b[i]*b[(i+1)%5] for i in range(5)) for b in bits])/np.sqrt(32)
def rho(k):
 psi=G*np.array([(-1)**sum(b[i]*((k>>i)&1) for i in range(5)) for b in bits]); return np.outer(psi,psi.conj())
r0=rho(0); rw=v*r0+(1-v)*np.eye(32)/32
pairs=[(0,8),(3,11),(0,9),(0,10)]
def assm(r, a,b,x,y,etaA=1.,etaB=1.):
 E=(P['I']+a*etaA*P['XY'[x]])/2; F=(P['I']+b*etaB*P['XY'[y]])/2
 z=(kron(E,F,np.eye(8))@r).reshape(4,8,4,8)
 return np.einsum('iaib->ab',z)
for a,b,x,y in product([1,-1],[1,-1],range(2),range(2)):
 mix=v/2*(assm(r0,a,b,x,y,c,1)+assm(r0,a,b,x,y,1,c))
 mix+=(1-v)/4*sum(assm((rho(i)+rho(j))/2,a,b,x,y) for i,j in pairs)
 assert np.max(np.abs(mix-assm(rw,a,b,x,y)))<1e-12
for (i,j),site,axis in zip(pairs,[3,3,4,2],'ZZXX'):
 U=kron(*[P[axis] if k==site else P['I'] for k in range(5)])
 assert np.allclose(U@rho(i)@U,rho(j))
print('All 16 conditional density operators and four single-site dephasing identities: PASS')


## Exact generator comparator and noise thresholds

In [ ]:
I,X,Z=P['I'],P['X'],P['Z']; cap=1+np.sqrt(5)
for theta in np.linspace(0,np.pi,301):
 s=np.sin(theta); d=np.cos(theta)
 H=kron(I,Z,I)+kron(I,I,Z)+s*kron(Z,I,I)+kron(X,I,X)+d*kron(X,X,I)
 for parity,idx in [(1,[0,3,5,6]),(-1,[1,2,4,7])]:
  vals=np.linalg.eigvalsh(H[np.ix_(idx,idx)])
  assert np.allclose(np.poly(vals),[1,0,-8,-8*parity*s,0],atol=1e-10)
  assert vals[-1]<=cap+1e-12
print('Generator parity polynomials (301 angles): PASS; analytic exact bound',2+np.sqrt(5))
print('Exact global visibility:',v,'white-noise tolerance:',1-v)
p=brentq(lambda p:((1-p)**3+4*(1-p)**4+3*(1-p)**5)/2-lam*(1-p)**3-2,0,.2)
print('Sufficient local noise tolerance:',p,'calibration coefficient:',9+3*lam)

# Check the manuscript's numerical endpoints and norm-error coefficient.
assert abs(v-.6306019374818707)<1e-12
assert abs(p-.09586633850517964)<1e-10
assert np.isclose(9+3*lam,3+6*np.sqrt(2))
trusted_weights=[sum(c!='I' for c in s[2:]) for s in ['XXYIY','XXXXX','XYZZY','XYIYX','YXYZZ','YXXYI','YYZIZ','YYIXI']]
assert sum(trusted_weights)==18
assert np.isclose(sum(trusted_weights)/2+3*lam,3+6*np.sqrt(2))
print('Exact visibility, sufficient local threshold and trusted-factor calibration coefficient: PASS')


## Reduced-check dense graph example

In [ ]:
if Path('random_graphs.csv').exists() and Path('sparse_checks.csv').exists():
 q=next(x for x in csv.DictReader(open('sparse_checks.csv')) if x['n']=='20')
 r=next(x for x in csv.DictReader(open('random_graphs.csv')) if x['n']==q['n'] and x['index']==q['sample_index'])
 adj=list(map(int,r['adjacency_masks'].split(';')))
 Q=list(map(int,q['generator_labels'].split(';')))
 def zmap(s):
  z=0
  while s:
   b=s&-s;z^=adj[b.bit_length()-1];s^=b
  return z
 qw=[(s|zmap(s)).bit_count() for s in Q]; mins=[99]*4; zz=[0]*(1<<20)
 for s in range(1,1<<20):
  b=s&-s;zz[s]=z=zz[s^b]^adj[b.bit_length()-1]
  if s&3==3: mins[z&3]=min(mins[z&3],(s|z).bit_count())
 f=lambda p:sum((1-p)**w for w in mins)-(4-2*np.sqrt(2))*sum(1-(1-p)**w for w in qw)-2*np.sqrt(2)
 threshold=brentq(f,0,.2)
 assert q['sample_index']=='0'
 assert qw==[9,14,17,16,14,16,12] and mins==[7,7,7,7]
 assert abs(threshold-.008635938148234)<1e-9
 print('Dense graph check weights:',qw,'minimum context weights:',mins,'penalty threshold:',threshold)
 print('Dense-graph manuscript example: PASS')
else:
 print('Generate random_graphs.csv and sparse_checks.csv first to reproduce the dense example.')
